# 03_Benchmarks
Classical 23 + CEC2017 + CEC2022

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
import time
import importlib.util
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/QSO_Research'

# ── Check compute environment ─────────────────────────────────────
try:
    import subprocess
    result = subprocess.run(['nvidia-smi'],
                           capture_output=True, text=True)
    print('✅ GPU runtime detected')
except FileNotFoundError:
    print('✅ CPU runtime detected')
    print('   All experiments will run on CPU')

import psutil
import multiprocessing
ram = psutil.virtual_memory()
print(f'💾 RAM:   {ram.total/1e9:.1f}GB total, '
      f'{ram.available/1e9:.1f}GB available')
print(f'🖥️  Cores: {multiprocessing.cpu_count()}')

print('\n' + '='*50)
print('NOTEBOOK 04 — BENCHMARK EXPERIMENTS')
print('='*50)
print('✅ Environment ready — proceed with Cell 2')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ CPU runtime detected
   All experiments will run on CPU
💾 RAM:   13.6GB total, 11.6GB available
🖥️  Cores: 2

NOTEBOOK 04 — BENCHMARK EXPERIMENTS
✅ Environment ready — proceed with Cell 2


In [28]:
# ── Load QSO from Drive ───────────────────────────────────────────
def load_qso():
    spec   = importlib.util.spec_from_file_location(
                 'qso', f'{BASE}/algorithms/qso.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.qso

qso_func = load_qso()

# ── Paste all competitor functions here ───────────────────────────
# Copy cells 2-14 from Notebook 02 here
# (shared utilities + all 14 competitor functions)
# This ensures everything runs in one session

def initialise_population(pop_size, dim, lb, ub, seed=42):
    np.random.seed(seed)
    lb = np.full(dim, lb) if np.isscalar(lb) else np.array(lb)
    ub = np.full(dim, ub) if np.isscalar(ub) else np.array(ub)
    X  = np.random.uniform(lb, ub, (pop_size, dim))
    return X, lb, ub

def evaluate_population(func, X):
    return np.array([func(X[i]) for i in range(len(X))])

def bound_check(X, lb, ub):
    return np.clip(X, lb, ub)

def get_best(fitness, X):
    idx = np.argmin(fitness)
    return fitness[idx], X[idx].copy()

print('✅ QSO loaded from Drive')
print('⚠️  Now paste all competitor functions from')
print('   Notebook 02 Cells 3-14 below this cell')
print('   Then run Cell 3 to verify all load correctly')

✅ QSO loaded from Drive
⚠️  Now paste all competitor functions from
   Notebook 02 Cells 3-14 below this cell
   Then run Cell 3 to verify all load correctly


In [29]:
def pso(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        w=0.7, c1=1.5, c2=1.5,
        seed=42):
    """
    Particle Swarm Optimisation (Kennedy & Eberhart, 1995)

    Parameters:
    -----------
    w  : float — inertia weight (default 0.7)
    c1 : float — cognitive coefficient (default 1.5)
    c2 : float — social coefficient (default 1.5)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)

    # Velocities
    V       = np.zeros((pop_size, dim))
    v_max   = 0.2 * (ub - lb)

    # Personal and global bests
    pbest_X = X.copy()
    fitness = evaluate_population(func, X)
    pbest_f = fitness.copy()

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        r1 = np.random.rand(pop_size, dim)
        r2 = np.random.rand(pop_size, dim)

        # Velocity update
        V = (w * V
             + c1 * r1 * (pbest_X - X)
             + c2 * r2 * (gbest_X  - X))
        V = np.clip(V, -v_max, v_max)

        # Position update
        X = X + V
        X = bound_check(X, lb, ub)

        # Fitness evaluation
        fitness = evaluate_population(func, X)

        # Update personal bests
        improved = fitness < pbest_f
        pbest_f[improved] = fitness[improved]
        pbest_X[improved] = X[improved]

        # Update global best
        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing PSO...')
from numpy import sum as npsum
sphere = lambda x: npsum(x**2)
f, _, c = pso(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'PSO did not improve'
print('✅ PSO operational')

Testing PSO...
  Sphere 30D: 1.1721e-05
✅ PSO operational


In [30]:
def ga(func, lb, ub, dim,
       pop_size=30, max_iter=500,
       cr=0.9, mr=0.01,
       seed=42):
    """
    Genetic Algorithm (Holland, 1992)

    Parameters:
    -----------
    cr : float — crossover rate (default 0.9)
    mr : float — mutation rate (default 0.01)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        new_X = np.zeros_like(X)

        for i in range(pop_size):
            # ── Tournament selection ──────────────────────────
            t1, t2 = np.random.randint(0, pop_size, 2)
            parent1 = X[t1] if fitness[t1] < fitness[t2] else X[t2]

            t3, t4 = np.random.randint(0, pop_size, 2)
            parent2 = X[t3] if fitness[t3] < fitness[t4] else X[t4]

            # ── Single-point crossover ────────────────────────
            if np.random.rand() < cr:
                point   = np.random.randint(1, dim)
                child   = np.concatenate([
                              parent1[:point],
                              parent2[point:]])
            else:
                child = parent1.copy()

            # ── Gaussian mutation ─────────────────────────────
            mask          = np.random.rand(dim) < mr
            child[mask]  += np.random.normal(
                                0, 0.1*(ub[mask]-lb[mask]))
            child         = np.clip(child, lb, ub)
            new_X[i]      = child

        X       = new_X
        fitness = evaluate_population(func, X)

        # Elite preservation — keep best from previous generation
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > gbest_f:
            X[worst_idx]       = gbest_X.copy()
            fitness[worst_idx] = gbest_f

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing GA...')
f, _, c = ga(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'GA did not improve'
print('✅ GA operational')

Testing GA...
  Sphere 30D: 6.9737e+00
✅ GA operational


In [31]:
def de(func, lb, ub, dim,
       pop_size=30, max_iter=500,
       F=0.5, cr=0.9,
       seed=42):
    """
    Differential Evolution (Storn & Price, 1997)
    DE/rand/1/bin variant.

    Parameters:
    -----------
    F  : float — scaling factor (default 0.5)
             Lower values (0.4-0.6) work better on
             continuous unimodal problems. Original
             paper recommends F in [0.4, 1.0].
    cr : float — crossover rate (default 0.9)

    Note on parameters:
    -------------------
    F=0.5 chosen based on parameter sensitivity analysis
    showing F=0.8 causes over-exploration on 30D continuous
    problems with pop_size=30 at 500 iterations.
    This is consistent with Storn & Price (1997) who note
    F in [0.4, 0.6] works well for most continuous problems.
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Mutation — DE/rand/1 ──────────────────────────
            idxs = list(range(pop_size))
            idxs.remove(i)
            a, b, c_idx = np.random.choice(idxs, 3, replace=False)

            mutant = X[a] + F * (X[b] - X[c_idx])
            mutant = np.clip(mutant, lb, ub)

            # ── Binomial crossover ────────────────────────────
            cross_mask = np.random.rand(dim) < cr
            # Guarantee at least one dimension crosses over
            cross_mask[np.random.randint(dim)] = True
            trial = np.where(cross_mask, mutant, X[i])

            # ── Greedy selection ──────────────────────────────
            trial_f = func(trial)
            if trial_f < fitness[i]:
                X[i]       = trial
                fitness[i] = trial_f
                if trial_f < gbest_f:
                    gbest_f = trial_f
                    gbest_X = trial.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing DE (fixed)...')
de_results = []
for seed in [42, 43, 44, 45, 46]:
    f, _, c = de(sphere, -100, 100, dim=30, seed=seed)
    de_results.append(f)
    improvement = (c[0] - f) / c[0] * 100
    print(f'  Seed {seed}: {f:.4e} '
          f'(improvement: {improvement:.1f}%)')

print(f'\n  Mean: {np.mean(de_results):.4e}')
print(f'  Std:  {np.std(de_results):.4e}')

# Convergence check
print('\n  Convergence check (seed 42):')
f, _, c = de(sphere, -100, 100, dim=30, seed=42)
checkpoints = [0, 50, 100, 200, 300, 400, 499]
for cp in checkpoints:
    print(f'    Iter {cp:3d}: {c[cp]:.4e}')

assert f < c[0], 'DE did not improve'
print('\n✅ DE (fixed) operational')

Testing DE (fixed)...
  Seed 42: 3.1691e-01 (improvement: 100.0%)
  Seed 43: 4.8467e+01 (improvement: 99.9%)
  Seed 44: 5.5768e-01 (improvement: 100.0%)
  Seed 45: 6.2047e-04 (improvement: 100.0%)
  Seed 46: 6.0036e+01 (improvement: 99.9%)

  Mean: 2.1876e+01
  Std:  2.6687e+01

  Convergence check (seed 42):
    Iter   0: 7.0687e+04
    Iter  50: 3.1347e+03
    Iter 100: 5.9598e+02
    Iter 200: 2.4100e+01
    Iter 300: 2.4555e+00
    Iter 400: 8.7099e-01
    Iter 499: 3.1914e-01

✅ DE (fixed) operational


In [32]:
def gwo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Grey Wolf Optimiser (Mirjalili et al., 2014)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    # Alpha, beta, delta wolves
    sorted_idx = np.argsort(fitness)
    alpha_f, alpha_X = fitness[sorted_idx[0]], X[sorted_idx[0]].copy()
    beta_f,  beta_X  = fitness[sorted_idx[1]], X[sorted_idx[1]].copy()
    delta_f, delta_X = fitness[sorted_idx[2]], X[sorted_idx[2]].copy()

    convergence = [alpha_f]

    for t in range(max_iter):
        # Linearly decreasing a from 2 to 0
        a = 2 - 2 * (t / max_iter)

        for i in range(pop_size):
            # Update position based on alpha, beta, delta
            X1 = _gwo_update(X[i], alpha_X, a)
            X2 = _gwo_update(X[i], beta_X,  a)
            X3 = _gwo_update(X[i], delta_X, a)
            X[i] = np.clip((X1 + X2 + X3) / 3, lb, ub)

        fitness = evaluate_population(func, X)

        # Update hierarchy
        sorted_idx = np.argsort(fitness)
        if fitness[sorted_idx[0]] < alpha_f:
            alpha_f = fitness[sorted_idx[0]]
            alpha_X = X[sorted_idx[0]].copy()
        if fitness[sorted_idx[1]] < beta_f:
            beta_f  = fitness[sorted_idx[1]]
            beta_X  = X[sorted_idx[1]].copy()
        if fitness[sorted_idx[2]] < delta_f:
            delta_f = fitness[sorted_idx[2]]
            delta_X = X[sorted_idx[2]].copy()

        convergence.append(alpha_f)

    return alpha_f, alpha_X, convergence


def _gwo_update(x, leader, a):
    """Helper — update position toward a leader wolf."""
    r1, r2 = np.random.rand(len(x)), np.random.rand(len(x))
    A = 2 * a * r1 - a
    C = 2 * r2
    D = np.abs(C * leader - x)
    return leader - A * D


# --- Test ---
print('Testing GWO...')
f, _, c = gwo(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'GWO did not improve'
print('✅ GWO operational')

Testing GWO...
  Sphere 30D: 1.3550e-31
✅ GWO operational


In [33]:
def woa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Whale Optimisation Algorithm (Mirjalili & Lewis, 2016)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        a  = 2 - 2 * (t / max_iter)  # Decreases from 2 to 0
        a2 = -1 - (t / max_iter)     # Decreases from -1 to -2

        for i in range(pop_size):
            r  = np.random.rand()
            A  = 2 * a * np.random.rand(dim) - a
            C  = 2 * np.random.rand(dim)
            b  = 1.0   # Spiral shape constant
            l  = (a2 - 1) * np.random.rand() + 1
            p  = np.random.rand()

            if p < 0.5:
                if np.linalg.norm(A) < 1:
                    # Shrinking encircling
                    D       = np.abs(C * gbest_X - X[i])
                    X[i]    = gbest_X - A * D
                else:
                    # Random search
                    rand_X  = X[np.random.randint(pop_size)]
                    D       = np.abs(C * rand_X - X[i])
                    X[i]    = rand_X - A * D
            else:
                # Spiral bubble-net attack
                D       = np.abs(gbest_X - X[i])
                X[i]    = (D * np.exp(b * l)
                           * np.cos(2 * np.pi * l)
                           + gbest_X)

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing WOA...')
f, _, c = woa(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'WOA did not improve'
print('✅ WOA operational')

Testing WOA...
  Sphere 30D: 3.7246e-07
✅ WOA operational


In [34]:
def sca(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Sine Cosine Algorithm (Mirjalili, 2016)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        # Decreasing r1 from 2 to 0
        r1 = 2 - 2 * (t / max_iter)

        for i in range(pop_size):
            r2 = 2 * np.pi * np.random.rand(dim)
            r3 = np.random.rand(dim)
            r4 = np.random.rand()

            if r4 < 0.5:
                X[i] = (X[i]
                        + r1 * np.sin(r2)
                        * np.abs(r3 * gbest_X - X[i]))
            else:
                X[i] = (X[i]
                        + r1 * np.cos(r2)
                        * np.abs(r3 * gbest_X - X[i]))

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing SCA...')
f, _, c = sca(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'SCA did not improve'
print('✅ SCA operational')

Testing SCA...
  Sphere 30D: 5.3338e-12
✅ SCA operational


In [35]:
def hho(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Harris Hawks Optimisation (Heidari et al., 2019)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        E0 = 2 * np.random.rand() - 1   # Initial energy
        E  = 2 * E0 * (1 - t / max_iter) # Escaping energy

        for i in range(pop_size):
            r = np.random.rand()

            if np.abs(E) >= 1:
                # ── Exploration ───────────────────────────────
                if r >= 0.5:
                    rand_X   = X[np.random.randint(pop_size)]
                    X[i]     = (rand_X
                                - np.random.rand()
                                * np.abs(rand_X
                                - 2 * np.random.rand() * X[i]))
                else:
                    X[i]     = ((gbest_X - np.mean(X, axis=0))
                                - np.random.rand()
                                * (lb + np.random.rand() * (ub - lb)))
            else:
                # ── Exploitation ──────────────────────────────
                J        = 2 * (1 - np.random.rand())
                delta_X  = gbest_X - X[i]

                if r >= 0.5 and np.abs(E) >= 0.5:
                    # Soft besiege
                    X[i] = delta_X - E * np.abs(J * gbest_X - X[i])

                elif r >= 0.5 and np.abs(E) < 0.5:
                    # Hard besiege
                    X[i] = gbest_X - E * np.abs(delta_X)

                elif r < 0.5 and np.abs(E) >= 0.5:
                    # Soft besiege with progressive rapid dives
                    Y = gbest_X - E * np.abs(J * gbest_X - X[i])
                    Z = Y + np.random.rand(dim) * _levy_hho(dim)
                    X[i] = (Y if func(Y) < func(Z) else Z)

                else:
                    # Hard besiege with progressive rapid dives
                    Y = gbest_X - E * np.abs(J * gbest_X - np.mean(X, axis=0))
                    Z = Y + np.random.rand(dim) * _levy_hho(dim)
                    X[i] = (Y if func(Y) < func(Z) else Z)

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def _levy_hho(dim, beta=1.5):
    """Lévy flight helper for HHO."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, dim)
    v = np.random.normal(0, 1, dim)
    return u / (np.abs(v)**(1/beta))


# --- Test ---
print('Testing HHO...')
f, _, c = hho(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'HHO did not improve'
print('✅ HHO operational')

Testing HHO...
  Sphere 30D: 1.4046e-84
✅ HHO operational


In [36]:
def mpa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Marine Predators Algorithm (Faramarzi et al., 2020)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)

    # Elite matrix — top predator
    Elite   = np.tile(gbest_X, (pop_size, 1))
    convergence = [gbest_f]
    P       = 0.5
    FADs    = 0.2

    for t in range(max_iter):
        CF = (1 - t/max_iter) ** (2*t/max_iter)

        RL = 0.05 * _levy_mpa(pop_size, dim)
        RB = np.random.randn(pop_size, dim)

        for i in range(pop_size):
            r  = np.random.rand()
            R  = np.random.rand(dim)

            if t < max_iter / 3:
                # Phase 1 — High velocity ratio (prey moves faster)
                stepsize   = RB[i] * (Elite[i] - RB[i] * X[i])
                X[i]      += P * stepsize

            elif t < 2 * max_iter / 3:
                if i < pop_size // 2:
                    # Phase 2a — Unit velocity ratio (Lévy)
                    stepsize = RL[i] * (Elite[i] - RL[i] * X[i])
                    X[i]    += P * stepsize
                else:
                    # Phase 2b — Unit velocity ratio (Brownian)
                    stepsize = RB[i] * (RB[i] * Elite[i] - X[i])
                    X[i]    += P * CF * stepsize
            else:
                # Phase 3 — Low velocity ratio (predator moves faster)
                stepsize   = RL[i] * (RL[i] * Elite[i] - X[i])
                X[i]      += P * CF * stepsize

            # FADs effect
            if np.random.rand() < FADs:
                U    = np.random.rand(dim) < FADs
                X[i]+= CF * (lb + np.random.rand(dim)*(ub-lb)) * U

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        # Update elite matrix
        Elite = np.tile(gbest_X, (pop_size, 1))
        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def _levy_mpa(n, d, beta=1.5):
    """Lévy flight helper for MPA."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, (n, d))
    v = np.random.normal(0, 1, (n, d))
    return u / (np.abs(v)**(1/beta))


# --- Test ---
print('Testing MPA...')
f, _, c = mpa(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'MPA did not improve'
print('✅ MPA operational')

Testing MPA...
  Sphere 30D: 1.4499e-02
✅ MPA operational


In [37]:
def bfo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        n_swim=4, n_tumble=4,
        seed=42):
    """
    Bacterial Foraging Optimisation (Passino, 2002)

    Parameters:
    -----------
    n_swim   : int — swim steps per chemotaxis (default 4)
    n_tumble : int — tumble steps (default 4)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    step_size = 0.1 * (ub - lb)
    iters_per_cycle = max(1, max_iter // (n_tumble * n_swim + 1))

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Tumble — random direction ─────────────────────
            delta = np.random.randn(dim)
            delta /= (np.linalg.norm(delta) + 1e-10)

            # ── Swim — move in tumble direction ───────────────
            for s in range(n_swim):
                X_new    = X[i] + step_size * delta
                X_new    = np.clip(X_new, lb, ub)
                f_new    = func(X_new)

                if f_new < fitness[i]:
                    X[i]       = X_new
                    fitness[i] = f_new
                    if f_new < gbest_f:
                        gbest_f = f_new
                        gbest_X = X_new.copy()
                else:
                    break

        # ── Reproduction — top half survives ─────────────────
        if t % iters_per_cycle == 0:
            sorted_idx   = np.argsort(fitness)
            X            = np.vstack([
                               X[sorted_idx[:pop_size//2]],
                               X[sorted_idx[:pop_size//2]]
                           ])
            fitness      = np.concatenate([
                               fitness[sorted_idx[:pop_size//2]],
                               fitness[sorted_idx[:pop_size//2]]
                           ])

        # Decrease step size over time
        step_size *= 0.99

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing BFO...')
f, _, c = bfo(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'BFO did not improve'
print('✅ BFO operational')

Testing BFO...
  Sphere 30D: 4.8163e-01
✅ BFO operational


In [38]:
def qbso(func, lb, ub, dim,
         pop_size=30, max_iter=500,
         qs_threshold=0.5,
         seed=42):
    """
    Quorum Sensing Bacterial Swarm Optimisation (QBSO)
    Based on: Li et al. (2019)

    QS used as enhancement to bacterial swarm —
    NOT as standalone framework (key distinction from QSO)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    step_size = 0.1 * (ub - lb)

    for t in range(max_iter):
        # ── Compute quorum signal ─────────────────────────────
        f_worst = np.max(fitness)
        f_best  = np.min(fitness)
        epsilon = 1e-10

        if f_worst - f_best < epsilon:
            qs_signal = 0.5
        else:
            qs_signal = np.mean(
                (f_worst - fitness) / (f_worst - f_best + epsilon))

        for i in range(pop_size):
            delta = np.random.randn(dim)
            delta /= (np.linalg.norm(delta) + 1e-10)

            if qs_signal >= qs_threshold:
                # QS triggered — move toward global best
                direction = gbest_X - X[i]
                norm      = np.linalg.norm(direction) + 1e-10
                X[i]     += step_size * (direction/norm)
            else:
                # QS not triggered — random walk
                X[i]     += step_size * delta

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        step_size *= 0.995
        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def qbho(func, lb, ub, dim,
         pop_size=30, max_iter=500,
         qs_threshold=0.5,
         seed=42):
    """
    Quorum Sensing Bacterial Horde Optimisation (QBHO)
    Based on: Alzaqebah et al. (2023)

    QS used to identify optimal bacterial positions —
    NOT as standalone framework (key distinction from QSO)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        # ── Quorum detection ──────────────────────────────────
        f_worst   = np.max(fitness)
        f_best    = np.min(fitness)
        epsilon   = 1e-10

        qs_signal = np.mean(
            (f_worst - fitness) / (f_worst - f_best + epsilon + 1e-10))

        # ── Worst position used as reference (per QBHO paper) ─
        worst_idx = np.argmax(fitness)

        for i in range(pop_size):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            if qs_signal >= qs_threshold:
                # Quorum active — avoid worst, move to best
                X[i] = (X[i]
                        + r1 * (gbest_X - X[i])
                        - r2 * (X[worst_idx] - X[i]))
            else:
                # Quorum inactive — standard foraging
                rand_X = X[np.random.randint(pop_size)]
                X[i]   = X[i] + r1 * (rand_X - X[i])

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Tests ---
print('Testing QBSO...')
f, _, c = qbso(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'QBSO did not improve'
print('✅ QBSO operational')

print('Testing QBHO...')
f, _, c = qbho(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'QBHO did not improve'
print('✅ QBHO operational')

Testing QBSO...
  Sphere 30D: 2.8946e+00
✅ QBSO operational
Testing QBHO...
  Sphere 30D: 8.8982e+03
✅ QBHO operational


In [39]:
def dbo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Dung Beetle Optimisation (Xue & Shen, 2022)

    Four beetle roles:
    - Ball-rollers  : navigate using celestial cues (exploration)
    - Dancers       : reorient when lost (escape local optima)
    - Foragers      : search near best site (exploitation)
    - Brood-stealers: compete for best positions (intensification)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    # Population split into 4 roles
    n_rollers  = pop_size // 4
    n_dancers  = pop_size // 4
    n_foragers = pop_size // 4
    n_thieves  = pop_size - n_rollers - n_dancers - n_foragers

    # Role index boundaries
    r_end = n_rollers
    d_end = n_rollers + n_dancers
    f_end = n_rollers + n_dancers + n_foragers

    for t in range(max_iter):
        R  = 1 - t / max_iter       # Decreasing radius
        CF = (1 - t/max_iter) ** 2  # Convergence factor

        # ── Ball-rolling beetles (exploration) ────────────────────
        for i in range(r_end):
            if np.random.rand() > 0.9:
                # Dancing reorientation
                X[i] = X[i] + np.tan(
                    np.random.rand(dim)) * np.abs(X[i] - gbest_X)
            else:
                # Navigate toward best with decreasing radius
                r1   = np.random.rand(dim)
                X[i] = X[i] + R * r1 * (gbest_X - X[i])
            X[i] = np.clip(X[i], lb, ub)

        # ── Dancing beetles (escape local optima) ─────────────────
        for i in range(r_end, d_end):
            r1   = np.random.rand(dim)
            X[i] = gbest_X + r1 * np.abs(X[i] - gbest_X) * CF
            X[i] = np.clip(X[i], lb, ub)

        # ── Foraging beetles (exploitation) — FIXED ───────────────
        for i in range(d_end, f_end):
            r1   = np.random.rand(dim)
            r2   = np.random.rand(dim)
            # Move toward global best with random perturbation
            X[i] = (X[i]
                    + r1 * (gbest_X - X[i])
                    + r2 * CF * np.random.randn(dim))
            X[i] = np.clip(X[i], lb, ub)

        # ── Brood-stealing beetles (intensification) ──────────────
        for i in range(f_end, pop_size):
            r1   = np.random.rand(dim)
            r2   = np.random.rand(dim)
            # Steal position near global best
            X[i] = (gbest_X
                    + r1 * CF * (X[i] - gbest_X)
                    + r2 * np.random.randn(dim) * R)
            X[i] = np.clip(X[i], lb, ub)

        # ── Evaluate & update best ────────────────────────────────
        fitness = evaluate_population(func, X)

        # Elite preservation
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > gbest_f:
            X[worst_idx]       = gbest_X.copy()
            fitness[worst_idx] = gbest_f

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
print('Testing DBO (fixed)...')
dbo_results = []
for seed in [42, 43, 44, 45, 46]:
    f, _, c = dbo(sphere, -100, 100, dim=30, seed=seed)
    dbo_results.append(f)
    improvement = (c[0] - f) / c[0] * 100
    print(f'  Seed {seed}: {f:.4e} '
          f'(improvement: {improvement:.1f}%)')

print(f'\n  Mean: {np.mean(dbo_results):.4e}')
print(f'  Std:  {np.std(dbo_results):.4e}')

# Convergence check
print('\n  Convergence check (seed 42):')
f, _, c = dbo(sphere, -100, 100, dim=30, seed=42)
checkpoints = [0, 50, 100, 200, 300, 400, 499]
for cp in checkpoints:
    print(f'    Iter {cp:3d}: {c[cp]:.4e}')

assert f < c[0], 'DBO did not improve'
assert np.mean(dbo_results) < 1e3, \
    f'DBO mean still too high: {np.mean(dbo_results):.4e}'
print('\n✅ DBO (fixed) operational')

Testing DBO (fixed)...
  Seed 42: 4.4626e-04 (improvement: 100.0%)
  Seed 43: 2.8123e-03 (improvement: 100.0%)
  Seed 44: 9.8802e-04 (improvement: 100.0%)
  Seed 45: 2.0863e-03 (improvement: 100.0%)
  Seed 46: 2.4295e-03 (improvement: 100.0%)

  Mean: 1.7525e-03
  Std:  8.9259e-04

  Convergence check (seed 42):
    Iter   0: 7.0687e+04
    Iter  50: 8.6470e+03
    Iter 100: 4.2550e+03
    Iter 200: 4.2591e+02
    Iter 300: 1.0321e+00
    Iter 400: 5.9660e-02
    Iter 499: 5.2092e-04

✅ DBO (fixed) operational


In [40]:
def poa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Pelican Optimisation Algorithm (Trojovský & Dehghani, 2022)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Phase 1: Moving toward prey ───────────────────
            # Random prey selection
            prey_idx  = np.random.randint(pop_size)
            prey_X    = X[prey_idx]
            prey_f    = fitness[prey_idx]

            X1 = X[i] + np.random.rand(dim) * (
                prey_X - np.random.randint(1, 3) * X[i])
            X1 = np.clip(X1, lb, ub)
            f1 = func(X1)

            if f1 < fitness[i]:
                X[i]       = X1
                fitness[i] = f1

            # ── Phase 2: Winging on water surface ─────────────
            R    = 0.2 * (1 - t / max_iter)
            X2   = X[i] + R * (2 * np.random.rand(dim) - 1) * X[i]
            X2   = np.clip(X2, lb, ub)
            f2   = func(X2)

            if f2 < fitness[i]:
                X[i]       = X2
                fitness[i] = f2

            if fitness[i] < gbest_f:
                gbest_f = fitness[i]
                gbest_X = X[i].copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def evo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Electric Eel Foraging Optimiser (EVO)
    Based on: Wang et al. (2024)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        a = 2 * (1 - t / max_iter)  # Decreasing factor

        for i in range(pop_size):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            # ── Electric discharge hunting ────────────────────
            if np.random.rand() < 0.5:
                # Discharge toward best
                X[i] = (X[i]
                        + a * r1 * (gbest_X - X[i])
                        + (1-a) * r2 * (
                            X[np.random.randint(pop_size)] - X[i]))
            else:
                # Passive drift with random component
                beta   = np.random.randn(dim)
                X[i]   = (gbest_X
                          + beta * np.abs(gbest_X - X[i]) * (1 - a))

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Tests ---
print('Testing POA...')
f, _, c = poa(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'POA did not improve'
print('✅ POA operational')

print('Testing EVO...')
f, _, c = evo(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'EVO did not improve'
print('✅ EVO operational')

Testing POA...
  Sphere 30D: 1.9721e-123
✅ POA operational
Testing EVO...
  Sphere 30D: 1.7627e+03
✅ EVO operational


In [41]:
def _levy_gjo(n, d, beta=1.5):
    """Lévy flight helper for GJO."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta
              * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, (n, d))
    v = np.random.normal(0, 1, (n, d))
    return u / (np.abs(v)**(1/beta))


def gjo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Golden Jackal Optimizer (Chopra & Ansari, 2022)
    Published: Expert Systems with Applications, 198, 116924

    Models male and female jackal hunting behaviour:
    - Male jackal: tracks prey (global best)
    - Female jackal: supports male (second best)
    - Prey escape energy decreases over iterations
    """
    X, lb, ub = initialise_population(
                    pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    # Male and female jackal (best two solutions)
    sorted_idx = np.argsort(fitness)
    male_pos   = X[sorted_idx[0]].copy()
    male_f     = fitness[sorted_idx[0]]
    female_pos = X[sorted_idx[1]].copy()
    female_f   = fitness[sorted_idx[1]]

    gbest_f     = male_f
    gbest_X     = male_pos.copy()
    convergence = [gbest_f]

    for t in range(max_iter):
        E1 = 1.5 * (1 - t / max_iter)
        RL = 0.05 * _levy_gjo(pop_size, dim)

        for i in range(pop_size):
            E0 = 2 * np.random.rand() - 1
            E  = E1 * E0

            # Update toward male jackal
            D_male   = np.abs(RL[i] * male_pos - X[i])
            X1       = male_pos - E * D_male

            # Update toward female jackal
            D_female = np.abs(RL[i] * female_pos - X[i])
            X2       = female_pos - E * D_female

            # Average of both updates
            X[i] = np.clip((X1 + X2) / 2, lb, ub)

        fitness = evaluate_population(func, X)

        # Update male and female jackals
        sorted_idx = np.argsort(fitness)

        if fitness[sorted_idx[0]] < male_f:
            male_f   = fitness[sorted_idx[0]]
            male_pos = X[sorted_idx[0]].copy()

        if fitness[sorted_idx[1]] < female_f:
            female_f   = fitness[sorted_idx[1]]
            female_pos = X[sorted_idx[1]].copy()

        if male_f < gbest_f:
            gbest_f = male_f
            gbest_X = male_pos.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---
sphere = lambda x: np.sum(x**2)

print('Testing GJO...')
f, _, c = gjo(sphere, -100, 100, dim=30, seed=42)
print(f'  Sphere 30D: {f:.4e}')
assert f < c[0], 'GJO did not improve'
print('✅ GJO operational')

Testing GJO...
  Sphere 30D: 4.7636e-134
✅ GJO operational


In [15]:
# ═══════════════════════════════════════════════════════
# PASTE ALL COMPETITOR FUNCTIONS FROM NOTEBOOK 02 HERE
# Cells 3 through 14 — all algorithm definitions
# PSO, GA, DE, GWO, WOA, SCA, HHO, MPA,
# BFO, QBSO, QBHO, DBO, POA, EVO
# ═══════════════════════════════════════════════════════

# After pasting, this cell should contain all 14
# competitor function definitions

print('✅ All competitor functions defined')

✅ All competitor functions defined


In [42]:
# ── Build unified registry ────────────────────────────────────────
ALGORITHM_REGISTRY = {
    'QSO':  lambda func, lb, ub, dim, seed:
                qso_func(func, lb, ub, dim,
                         pop_size=30, max_iter=500,
                         theta_min=0.3, theta_max=0.7,
                         lambda_=0.05, tau=10, alpha=1.25,
                         seed=seed),
    'PSO':  lambda func, lb, ub, dim, seed:
                pso(func, lb, ub, dim,
                    pop_size=30, max_iter=500,
                    w=0.7, c1=1.5, c2=1.5, seed=seed),
    'GA':   lambda func, lb, ub, dim, seed:
                ga(func, lb, ub, dim,
                   pop_size=30, max_iter=500,
                   cr=0.9, mr=0.01, seed=seed),
    'DE':   lambda func, lb, ub, dim, seed:
                de(func, lb, ub, dim,
                   pop_size=30, max_iter=500,
                   F=0.5, cr=0.9, seed=seed),
    'GWO':  lambda func, lb, ub, dim, seed:
                gwo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'WOA':  lambda func, lb, ub, dim, seed:
                woa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'SCA':  lambda func, lb, ub, dim, seed:
                sca(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'HHO':  lambda func, lb, ub, dim, seed:
                hho(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'MPA':  lambda func, lb, ub, dim, seed:
                mpa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'BFO':  lambda func, lb, ub, dim, seed:
                bfo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'QBSO': lambda func, lb, ub, dim, seed:
                qbso(func, lb, ub, dim,
                     pop_size=30, max_iter=500, seed=seed),
    'QBHO': lambda func, lb, ub, dim, seed:
                qbho(func, lb, ub, dim,
                     pop_size=30, max_iter=500, seed=seed),
    'DBO':  lambda func, lb, ub, dim, seed:
                dbo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'POA':  lambda func, lb, ub, dim, seed:
                poa(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
    'EVO':  lambda func, lb, ub, dim, seed:
                evo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),

    'GJO':  lambda func, lb, ub, dim, seed:
                gjo(func, lb, ub, dim,
                    pop_size=30, max_iter=500, seed=seed),
}

def run_algorithm(algo_name, func, lb, ub, dim, seed):
    """Unified runner — returns (best_f, best_x, convergence)."""
    result = ALGORITHM_REGISTRY[algo_name](
                 func, lb, ub, dim, seed)
    return result[0], result[1], result[2]

# ── Quick verification ────────────────────────────────────────────
print('Verifying all algorithms load correctly...')
sphere = lambda x: np.sum(x**2)
errors = []
for algo in ALGORITHM_REGISTRY:
    try:
        f, _, _ = run_algorithm(algo, sphere, -100, 100,
                                dim=10, seed=42)
        print(f'  ✅ {algo:<6}: {f:.4e}')
    except Exception as e:
        print(f'  ❌ {algo:<6}: {e}')
        errors.append(algo)

if errors:
    print(f'\n⚠️  Fix these before proceeding: {errors}')
else:
    print(f'\n✅ All {len(ALGORITHM_REGISTRY)} algorithms ready')

Verifying all algorithms load correctly...
  ✅ QSO   : 2.7269e-07
  ✅ PSO   : 4.0302e-26
  ✅ GA    : 2.4068e-01
  ✅ DE    : 3.3529e-26
  ✅ GWO   : 1.6789e-64
  ✅ WOA   : 3.3734e-19
  ✅ SCA   : 2.0750e-30
  ✅ HHO   : 3.8324e-89
  ✅ MPA   : 2.6528e-09
  ✅ BFO   : 1.0164e-02
  ✅ QBSO  : 7.3820e-02
  ✅ QBHO  : 3.3231e+02
  ✅ DBO   : 2.9809e-06
  ✅ POA   : 1.1152e-140
  ✅ EVO   : 3.5217e-03
  ✅ GJO   : 4.6244e-129

✅ All 16 algorithms ready


In [43]:
def make_classical_functions():
    """
    Classical 23 benchmark functions.
    Standard suite used in metaheuristic literature.

    F1-F7:   Unimodal functions
    F8-F13:  Multimodal functions (many local optima)
    F14-F23: Fixed-dimension multimodal functions

    Reference: Yang (2010), Nature-Inspired Metaheuristic
    Algorithms.
    """
    funcs = {}

    # ── F1: Sphere ───────────────────────────────────────────
    funcs['F1'] = {
        'func': lambda x: np.sum(x**2),
        'lb': -100, 'ub': 100, 'dim': 30,
        'optimum': 0, 'type': 'Unimodal'
    }

    # ── F2: Schwefel 2.22 ─────────────────────────────────────
    funcs['F2'] = {
        'func': lambda x: np.sum(np.abs(x)) + np.prod(np.abs(x)),
        'lb': -10, 'ub': 10, 'dim': 30,
        'optimum': 0, 'type': 'Unimodal'
    }

    # ── F3: Schwefel 1.2 ──────────────────────────────────────
    funcs['F3'] = {
        'func': lambda x: np.sum([np.sum(x[:i+1])**2
                                   for i in range(len(x))]),
        'lb': -100, 'ub': 100, 'dim': 30,
        'optimum': 0, 'type': 'Unimodal'
    }

    # ── F4: Schwefel 2.21 ─────────────────────────────────────
    funcs['F4'] = {
        'func': lambda x: np.max(np.abs(x)),
        'lb': -100, 'ub': 100, 'dim': 30,
        'optimum': 0, 'type': 'Unimodal'
    }

    # ── F5: Rosenbrock ────────────────────────────────────────
    funcs['F5'] = {
        'func': lambda x: np.sum(
            100*(x[1:]-x[:-1]**2)**2 + (x[:-1]-1)**2),
        'lb': -30, 'ub': 30, 'dim': 30,
        'optimum': 0, 'type': 'Unimodal'
    }

    # ── F6: Step ──────────────────────────────────────────────
    funcs['F6'] = {
        'func': lambda x: np.sum(np.floor(x + 0.5)**2),
        'lb': -100, 'ub': 100, 'dim': 30,
        'optimum': 0, 'type': 'Unimodal'
    }

    # ── F7: Quartic with noise ────────────────────────────────
    funcs['F7'] = {
        'func': lambda x: np.sum(
            np.arange(1, len(x)+1) * x**4) + np.random.rand(),
        'lb': -1.28, 'ub': 1.28, 'dim': 30,
        'optimum': 0, 'type': 'Unimodal'
    }

    # ── F8: Schwefel 2.26 ─────────────────────────────────────
    funcs['F8'] = {
        'func': lambda x: -np.sum(
            x * np.sin(np.sqrt(np.abs(x)))),
        'lb': -500, 'ub': 500, 'dim': 30,
        'optimum': -12569.5, 'type': 'Multimodal'
    }

    # ── F9: Rastrigin ─────────────────────────────────────────
    funcs['F9'] = {
        'func': lambda x: 10*len(x) + np.sum(
            x**2 - 10*np.cos(2*np.pi*x)),
        'lb': -5.12, 'ub': 5.12, 'dim': 30,
        'optimum': 0, 'type': 'Multimodal'
    }

    # ── F10: Ackley ───────────────────────────────────────────
    funcs['F10'] = {
        'func': lambda x: (
            -20*np.exp(-0.2*np.sqrt(np.sum(x**2)/len(x)))
            - np.exp(np.sum(np.cos(2*np.pi*x))/len(x))
            + 20 + np.e),
        'lb': -32, 'ub': 32, 'dim': 30,
        'optimum': 0, 'type': 'Multimodal'
    }

    # ── F11: Griewank ─────────────────────────────────────────
    funcs['F11'] = {
        'func': lambda x: (
            np.sum(x**2)/4000
            - np.prod(np.cos(x/np.sqrt(
                np.arange(1, len(x)+1)))) + 1),
        'lb': -600, 'ub': 600, 'dim': 30,
        'optimum': 0, 'type': 'Multimodal'
    }

    # ── F12: Penalised 1 ──────────────────────────────────────
    def _u(x, a, k, m):
        return np.where(x > a,  k*(x-a)**m,
               np.where(x < -a, k*(-x-a)**m, 0))

    def _f12(x):
        d  = len(x)
        y  = 1 + (x + 1) / 4
        s  = (np.pi/d) * (
            10*np.sin(np.pi*y[0])**2
            + np.sum((y[:-1]-1)**2
            * (1+10*np.sin(np.pi*y[1:])**2))
            + (y[-1]-1)**2)
        return s + np.sum(_u(x, 10, 100, 4))

    funcs['F12'] = {
        'func': _f12,
        'lb': -50, 'ub': 50, 'dim': 30,
        'optimum': 0, 'type': 'Multimodal'
    }

    # ── F13: Penalised 2 ──────────────────────────────────────
    def _f13(x):
        d = len(x)
        s = (0.1 * (
            np.sin(3*np.pi*x[0])**2
            + np.sum((x[:-1]-1)**2
            * (1+np.sin(3*np.pi*x[1:])**2))
            + (x[-1]-1)**2*(1+np.sin(2*np.pi*x[-1])**2)))
        return s + np.sum(_u(x, 5, 100, 4))

    funcs['F13'] = {
        'func': _f13,
        'lb': -50, 'ub': 50, 'dim': 30,
        'optimum': 0, 'type': 'Multimodal'
    }

    # ── F14: Shekel's Foxholes ────────────────────────────────
    _a14 = np.array([[-32,-16,0,16,32]*5,
                     [-32,-32,-32,-32,-32,
                      -16,-16,-16,-16,-16,
                        0,  0,  0,  0,  0,
                       16, 16, 16, 16, 16,
                       32, 32, 32, 32, 32]])
    def _f14(x):
        s = 0
        for j in range(25):
            s += 1.0/(j+1 + np.sum((x[:2]-_a14[:,j])**2))
        return (1/500 + s)**(-1)

    funcs['F14'] = {
        'func': _f14,
        'lb': -65.536, 'ub': 65.536, 'dim': 2,
        'optimum': 1, 'type': 'Fixed-dim'
    }

    # ── F15: Kowalik ──────────────────────────────────────────
    _b15 = np.array([0.1957,0.1947,0.1735,0.1600,0.0844,
                     0.0627,0.0456,0.0342,0.0323,0.0235,0.0246])
    _a15 = np.array([4,2,1,0.5,0.25,0.1667,0.125,0.1,
                     0.0833,0.0714,0.0625])
    def _f15(x):
        return np.sum(
            (_b15 - x[0]*((_a15**2+_a15*x[1])
            / (_a15**2+_a15*x[2]+x[3])))**2)

    funcs['F15'] = {
        'func': _f15,
        'lb': -5, 'ub': 5, 'dim': 4,
        'optimum': 0.00030748, 'type': 'Fixed-dim'
    }

    # ── F16: Six-Hump Camel ───────────────────────────────────
    funcs['F16'] = {
        'func': lambda x: (
            4*x[0]**2 - 2.1*x[0]**4 + x[0]**6/3
            + x[0]*x[1] - 4*x[1]**2 + 4*x[1]**4),
        'lb': -5, 'ub': 5, 'dim': 2,
        'optimum': -1.0316, 'type': 'Fixed-dim'
    }

    # ── F17: Branin ───────────────────────────────────────────
    funcs['F17'] = {
        'func': lambda x: (
            (x[1]-5.1/(4*np.pi**2)*x[0]**2
             + 5/np.pi*x[0]-6)**2
            + 10*(1-1/(8*np.pi))*np.cos(x[0])+10),
        'lb': -5, 'ub': 15, 'dim': 2,
        'optimum': 0.398, 'type': 'Fixed-dim'
    }

    # ── F18: Goldstein-Price ──────────────────────────────────
    funcs['F18'] = {
        'func': lambda x: (
            (1+(x[0]+x[1]+1)**2
             * (19-14*x[0]+3*x[0]**2
                -14*x[1]+6*x[0]*x[1]+3*x[1]**2))
            * (30+(2*x[0]-3*x[1])**2
               * (18-32*x[0]+12*x[0]**2
                  +48*x[1]-36*x[0]*x[1]+27*x[1]**2))),
        'lb': -2, 'ub': 2, 'dim': 2,
        'optimum': 3, 'type': 'Fixed-dim'
    }

    # ── F19: Hartman 3D ───────────────────────────────────────
    _a19 = np.array([[3,10,30],[0.1,10,35],
                     [3,10,30],[0.1,10,35]])
    _c19 = np.array([1,1.2,3,3.2])
    _p19 = np.array([[0.3689,0.1170,0.2673],
                     [0.4699,0.4387,0.7470],
                     [0.1091,0.8732,0.5547],
                     [0.0381,0.5743,0.8828]])
    def _f19(x):
        return -np.sum(_c19 * np.exp(
            -np.sum(_a19*(x-_p19)**2, axis=1)))

    funcs['F19'] = {
        'func': _f19,
        'lb': 0, 'ub': 1, 'dim': 3,
        'optimum': -3.86, 'type': 'Fixed-dim'
    }

    # ── F20: Hartman 6D ───────────────────────────────────────
    _a20 = np.array([[10,3,17,3.5,1.7,8],
                     [0.05,10,17,0.1,8,14],
                     [3,3.5,1.7,10,17,8],
                     [17,8,0.05,10,0.1,14]])
    _c20 = np.array([1,1.2,3,3.2])
    _p20 = np.array([[0.1312,0.1696,0.5569,0.0124,0.8283,0.5886],
                     [0.2329,0.4135,0.8307,0.3736,0.1004,0.9991],
                     [0.2348,0.1451,0.3522,0.2883,0.3047,0.6650],
                     [0.4047,0.8828,0.8732,0.5743,0.1091,0.0381]])
    def _f20(x):
        return -np.sum(_c20 * np.exp(
            -np.sum(_a20*(x-_p20)**2, axis=1)))

    funcs['F20'] = {
        'func': _f20,
        'lb': 0, 'ub': 1, 'dim': 6,
        'optimum': -3.32, 'type': 'Fixed-dim'
    }

    # ── F21: Shekel 5 ─────────────────────────────────────────
    _a21 = np.array([[4,4,4,4],[1,1,1,1],[8,8,8,8],
                     [6,6,6,6],[3,7,3,7]])
    _c21 = np.array([0.1,0.2,0.2,0.4,0.4])
    def _f21(x):
        return -np.sum(1/(np.sum(
            (x-_a21)**2, axis=1)+_c21))

    funcs['F21'] = {
        'func': _f21,
        'lb': 0, 'ub': 10, 'dim': 4,
        'optimum': -10.1532, 'type': 'Fixed-dim'
    }

    # ── F22: Shekel 7 ─────────────────────────────────────────
    _a22 = np.array([[4,4,4,4],[1,1,1,1],[8,8,8,8],
                     [6,6,6,6],[3,7,3,7],[2,9,2,9],[5,5,3,3]])
    _c22 = np.array([0.1,0.2,0.2,0.4,0.4,0.6,0.3])
    def _f22(x):
        return -np.sum(1/(np.sum(
            (x-_a22)**2, axis=1)+_c22))

    funcs['F22'] = {
        'func': _f22,
        'lb': 0, 'ub': 10, 'dim': 4,
        'optimum': -10.4028, 'type': 'Fixed-dim'
    }

    # ── F23: Shekel 10 ────────────────────────────────────────
    _a23 = np.array([[4,4,4,4],[1,1,1,1],[8,8,8,8],
                     [6,6,6,6],[3,7,3,7],[2,9,2,9],
                     [5,5,3,3],[8,1,8,1],[6,2,6,2],[7,3.6,7,3.6]])
    _c23 = np.array([0.1,0.2,0.2,0.4,0.4,0.6,0.3,0.7,0.5,0.5])
    def _f23(x):
        return -np.sum(1/(np.sum(
            (x-_a23)**2, axis=1)+_c23))

    funcs['F23'] = {
        'func': _f23,
        'lb': 0, 'ub': 10, 'dim': 4,
        'optimum': -10.5363, 'type': 'Fixed-dim'
    }

    return funcs


CLASSICAL_FUNCS = make_classical_functions()
print(f'✅ Classical benchmark suite loaded')
print(f'   {len(CLASSICAL_FUNCS)} functions: F1-F23')
print(f'   Types: Unimodal (F1-F7), Multimodal (F8-F13),')
print(f'          Fixed-dim (F14-F23)')

✅ Classical benchmark suite loaded
   23 functions: F1-F23
   Types: Unimodal (F1-F7), Multimodal (F8-F13),
          Fixed-dim (F14-F23)


In [44]:
## cell 5 ##

def evaluate_population_fast(func, X):
    """
    Faster population evaluation using list comprehension.
    Marginally faster than the numpy array approach.
    """
    return np.fromiter(
        (func(X[i]) for i in range(len(X))),
        dtype=float,
        count=len(X)
    )


def run_single_experiment(algo_name, func_info,
                           func_id, seed):
    """
    Run one algorithm on one function with one seed.
    Returns minimal data to preserve memory.
    """
    func = func_info['func']
    lb   = func_info['lb']
    ub   = func_info['ub']
    dim  = func_info['dim']

    try:
        best_f, _, conv = run_algorithm(
            algo_name, func, lb, ub, dim, seed)
        return {
            'algo':   algo_name,
            'func':   func_id,
            'seed':   seed,
            'best':   float(best_f),
            'conv':   [float(v) for v in conv],
            'status': 'ok'
        }
    except Exception as e:
        return {
            'algo':   algo_name,
            'func':   func_id,
            'seed':   seed,
            'best':   float('inf'),
            'conv':   [],
            'status': f'error: {e}'
        }


def run_benchmark_suite(funcs_dict, algo_list,
                         seeds, save_path,
                         suite_name='benchmark'):
    """
    Run complete benchmark suite with checkpointing.
    Saves results after each algorithm completes.

    Parameters:
    -----------
    funcs_dict  : dict — benchmark functions
    algo_list   : list — algorithm names to run
    seeds       : list — random seeds (30 for paper)
    save_path   : str  — Drive path for results
    suite_name  : str  — name for progress reporting
    """
    os.makedirs(save_path, exist_ok=True)
    all_results = []

    total = len(algo_list) * len(funcs_dict) * len(seeds)
    completed = 0
    start_time = time.time()

    print(f'\n{"="*55}')
    print(f'{suite_name} — Starting')
    print(f'  Algorithms : {len(algo_list)}')
    print(f'  Functions  : {len(funcs_dict)}')
    print(f'  Seeds      : {len(seeds)}')
    print(f'  Total runs : {total}')
    print(f'{"="*55}\n')

    for algo in algo_list:
        algo_results = []
        algo_start   = time.time()
        print(f'Running {algo}...', end=' ', flush=True)

        for func_id, func_info in funcs_dict.items():
            for seed in seeds:
                result = run_single_experiment(
                    algo, func_info, func_id, seed)
                algo_results.append(result)
                all_results.append(result)
                completed += 1

        # ── Checkpoint save after each algorithm ──────────────
        chk_path = f'{save_path}/checkpoint_{algo}.json'
        with open(chk_path, 'w') as f:
            json.dump(algo_results, f)

        algo_time = time.time() - algo_start
        elapsed   = time.time() - start_time
        remaining = (elapsed / completed) * (total - completed)

        print(f'✅ ({algo_time:.1f}s) | '
              f'Elapsed: {elapsed/60:.1f}m | '
              f'ETA: {remaining/60:.1f}m')

    # ── Save complete results ─────────────────────────────────
    results_path = f'{save_path}/all_results.json'
    with open(results_path, 'w') as f:
        json.dump(all_results, f)

    total_time = time.time() - start_time
    print(f'\n✅ {suite_name} complete')
    print(f'   Total time: {total_time/60:.1f} minutes')
    print(f'   Results saved: {results_path}')

    return all_results


print('✅ Experiment runner defined')

✅ Experiment runner defined


In [45]:
def compile_results(results_list, funcs_dict, algo_list):
    """
    Compile raw results into summary statistics.

    Returns:
    --------
    summary : dict — {algo: {func: {mean, std, best, worst}}}
    conv_data : dict — convergence curves for plotting
    """
    summary   = {algo: {} for algo in algo_list}
    conv_data = {algo: {} for algo in algo_list}

    for algo in algo_list:
        for func_id in funcs_dict:
            # Get all runs for this algo-function pair
            runs = [r['best'] for r in results_list
                    if r['algo'] == algo
                    and r['func'] == func_id
                    and r['status'] == 'ok']

            convs = [r['conv'] for r in results_list
                     if r['algo'] == algo
                     and r['func'] == func_id
                     and r['status'] == 'ok'
                     and len(r['conv']) > 0]

            if runs:
                summary[algo][func_id] = {
                    'mean':   float(np.mean(runs)),
                    'std':    float(np.std(runs)),
                    'best':   float(np.min(runs)),
                    'worst':  float(np.max(runs)),
                    'median': float(np.median(runs)),
                    'runs':   runs,
                }

            if convs:
                # Mean convergence curve
                min_len = min(len(c) for c in convs)
                conv_arr = np.array([c[:min_len] for c in convs])
                conv_data[algo][func_id] = \
                    np.mean(conv_arr, axis=0).tolist()

    return summary, conv_data


def results_to_dataframe(summary, funcs_dict, algo_list):
    """
    Convert summary to pandas DataFrame for easy export.
    Format: rows=functions, cols=algorithms (mean±std)
    """
    rows = []
    for func_id in funcs_dict:
        row = {'Function': func_id,
               'Type': funcs_dict[func_id]['type']}
        for algo in algo_list:
            if func_id in summary[algo]:
                m = summary[algo][func_id]['mean']
                s = summary[algo][func_id]['std']
                row[algo] = f'{m:.4e}±{s:.2e}'
                row[f'{algo}_mean'] = m
                row[f'{algo}_std']  = s
            else:
                row[algo] = 'N/A'
        rows.append(row)

    return pd.DataFrame(rows)


print('✅ Results compiler defined')

✅ Results compiler defined


In [ ]:
# ── UPDATED CELL 7 — Resume-capable ───────────────────────────────
SEEDS     = list(range(42, 72))
ALGO_LIST = list(ALGORITHM_REGISTRY.keys())
SAVE_PATH = f'{BASE}/results/raw/classical'

# Check what's already done
completed = get_completed_algos(SAVE_PATH)
remaining = [a for a in ALGO_LIST if a not in completed]

print('Classical 23 — RESUME MODE')
print(f'\nAlready completed ({len(completed)}):')
for a in completed:
    print(f'  ✅ {a}')
print(f'\nStill to run ({len(remaining)}):')
for a in remaining:
    print(f'  ⏳ {a}')
print(f'\nTotal remaining runs: '
      f'{len(remaining)*23*30}')

# Run remaining only
classical_results = run_benchmark_resume(
    funcs_dict = CLASSICAL_FUNCS,
    algo_list  = ALGO_LIST,
    seeds      = SEEDS,
    save_path  = SAVE_PATH,
    suite_name = 'Classical 23 — Resume'
)

# Compile all results
classical_summary, classical_conv = compile_results(
    classical_results, CLASSICAL_FUNCS, ALGO_LIST)

with open(f'{SAVE_PATH}/summary.json', 'w') as f:
    json.dump(classical_summary, f, indent=2)

print('\n✅ Classical 23 complete and saved')

Classical 23 — RESUME MODE

Already completed (11):
  ✅ QSO
  ✅ PSO
  ✅ GA
  ✅ DE
  ✅ GWO
  ✅ WOA
  ✅ SCA
  ✅ HHO
  ✅ MPA
  ✅ BFO
  ✅ QBSO

Still to run (4):
  ⏳ QBHO
  ⏳ DBO
  ⏳ POA
  ⏳ EVO

Total remaining runs: 2760
Previously completed (11):
  ✅ QSO
  ✅ PSO
  ✅ GA
  ✅ DE
  ✅ GWO
  ✅ WOA
  ✅ SCA
  ✅ HHO
  ✅ MPA
  ✅ BFO
  ✅ QBSO

Remaining (4):
  ⏳ QBHO
  ⏳ DBO
  ⏳ POA
  ⏳ EVO


Classical 23 — Resume — Starting
  Algorithms : 4
  Functions  : 23
  Seeds      : 30
  Total runs : 2760

Running QBHO... ✅ (414.7s) | Elapsed: 6.9m | ETA: 20.7m
Running DBO... ✅ (423.5s) | Elapsed: 14.0m | ETA: 14.0m
Running POA... ✅ (858.1s) | Elapsed: 28.3m | ETA: 9.4m
Running EVO... ✅ (455.2s) | Elapsed: 35.9m | ETA: 0.0m

✅ Classical 23 — Resume complete
   Total time: 35.9 minutes
   Results saved: /content/drive/MyDrive/QSO_Research/results/raw/classical/all_results.json

✅ All results merged and saved

✅ Classical 23 complete and saved


In [46]:
### CELL 7b #######

# ── Resume-capable runner ─────────────────────────────────────────
def get_completed_algos(save_path):
    completed = []
    for algo in ALGO_LIST:
        chk = f'{save_path}/checkpoint_{algo}.json'
        if os.path.exists(chk):
            completed.append(algo)
    return completed

def load_all_checkpoints(save_path, algo_list):
    all_results = []
    for algo in algo_list:
        chk = f'{save_path}/checkpoint_{algo}.json'
        if os.path.exists(chk):
            with open(chk, 'r') as f:
                all_results.extend(json.load(f))
    return all_results

def run_benchmark_resume(funcs_dict, algo_list,
                          seeds, save_path,
                          suite_name='benchmark'):
    """
    Resume-capable benchmark runner.
    Skips algorithms that already have checkpoints.
    """
    os.makedirs(save_path, exist_ok=True)
    completed = get_completed_algos(save_path)
    remaining = [a for a in algo_list
                 if a not in completed]

    if completed:
        print(f'Previously completed ({len(completed)}):')
        for a in completed:
            print(f'  ✅ {a}')
        print(f'\nRemaining ({len(remaining)}):')
        for a in remaining:
            print(f'  ⏳ {a}')
        print()

    if not remaining:
        print('✅ All algorithms already completed!')
        print('   Loading from checkpoints...')
        return load_all_checkpoints(save_path, algo_list)

    # Run remaining algorithms only
    new_results = run_benchmark_suite(
        funcs_dict  = funcs_dict,
        algo_list   = remaining,
        seeds       = seeds,
        save_path   = save_path,
        suite_name  = suite_name
    )

    # Merge with already completed results
    all_results = load_all_checkpoints(save_path, algo_list)

    # Save merged complete results
    results_path = f'{save_path}/all_results.json'
    with open(results_path, 'w') as f:
        json.dump(all_results, f)

    print(f'\n✅ All results merged and saved')
    return all_results

print('✅ Resume runner ready')
print('   Use run_benchmark_resume() for Cells 9 and 10')

✅ Resume runner ready
   Use run_benchmark_resume() for Cells 9 and 10


In [47]:
##### CELL 8 ####

def make_cec2017_functions(dim=30):
    """
    CEC 2017 benchmark suite — manual implementation.

    Reference: Awad et al. (2016) Problem Definitions and
    Evaluation Criteria for the CEC 2017 Special Session
    and Competition on Single Objective Bound Constrained
    Real-Parameter Numerical Optimization.

    F1:      Shifted and Rotated Bent Cigar
    F3:      Shifted and Rotated Zakharov
    F4-F10:  Multimodal functions
    F11-F20: Hybrid functions
    F21-F29: Composition functions

    Note: F2 excluded (deprecated in CEC 2017 erratum)

    For competition-grade implementations, all functions
    use fixed shift vectors and rotation matrices seeded
    for reproducibility.
    """
    np.random.seed(2017)  # Fixed seed for CEC 2017

    # ── Generate shift vectors and rotation matrices ───────────────
    shifts   = {i: np.random.uniform(-80, 80, dim)
                for i in range(1, 30)}
    rotations = {i: _make_rotation(dim)
                 for i in range(1, 30)}

    funcs = {}

    # ── F1: Bent Cigar ────────────────────────────────────────
    def f1(x):
        z = rotations[1] @ (x - shifts[1])
        return z[0]**2 + 1e6 * np.sum(z[1:]**2)

    funcs['F1'] = {
        'func': f1, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Unimodal'
    }

    # ── F3: Zakharov ──────────────────────────────────────────
    def f3(x):
        z = rotations[3] @ (x - shifts[3])
        i = np.arange(1, dim+1)
        s = np.sum(z**2)
        t = np.sum(0.5*i*z)
        return s + t**2 + t**4

    funcs['F3'] = {
        'func': f3, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Unimodal'
    }

    # ── F4: Rosenbrock ────────────────────────────────────────
    def f4(x):
        z = rotations[4] @ (x - shifts[4]) + 1
        return np.sum(100*(z[:-1]**2-z[1:])**2 + (z[:-1]-1)**2)

    funcs['F4'] = {
        'func': f4, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F5: Rastrigin ─────────────────────────────────────────
    def f5(x):
        z = rotations[5] @ (x - shifts[5])
        return 10*dim + np.sum(z**2 - 10*np.cos(2*np.pi*z))

    funcs['F5'] = {
        'func': f5, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F6: Expanded Scaffer ──────────────────────────────────
    def _scaffer(x, y):
        return (0.5 + (np.sin(np.sqrt(x**2+y**2))**2-0.5)
                / (1+0.001*(x**2+y**2))**2)

    def f6(x):
        z = rotations[6] @ (x - shifts[6])
        return np.sum([_scaffer(z[i], z[(i+1)%dim])
                       for i in range(dim)])

    funcs['F6'] = {
        'func': f6, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F7: Levy ──────────────────────────────────────────────
    def f7(x):
        z = rotations[7] @ (x - shifts[7])
        w = 1 + (z-1)/4
        return (np.sin(np.pi*w[0])**2
                + np.sum((w[:-1]-1)**2
                * (1+10*np.sin(np.pi*w[:-1]+1)**2))
                + (w[-1]-1)**2*(1+np.sin(2*np.pi*w[-1])**2))

    funcs['F7'] = {
        'func': f7, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F8: Modified Schwefel ─────────────────────────────────
    def f8(x):
        z = rotations[8] @ (x - shifts[8])
        z = z + 4.209687462275036e+002
        s = 0
        for zi in z:
            if np.abs(zi) <= 500:
                s -= zi*np.sin(np.abs(zi)**0.5)
            elif zi > 500:
                s += (zi-500)*np.sin(np.abs(500)**0.5) \
                     + (zi-500)**2/(10000*dim)
            else:
                s += (zi+500)*np.sin(np.abs(500)**0.5) \
                     + (zi+500)**2/(10000*dim)
        return s + 4.189828872724338e+002*dim

    funcs['F8'] = {
        'func': f8, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F9: Ackley ────────────────────────────────────────────
    def f9(x):
        z = rotations[9] @ (x - shifts[9])
        return (-20*np.exp(-0.2*np.sqrt(np.sum(z**2)/dim))
                - np.exp(np.sum(np.cos(2*np.pi*z))/dim)
                + 20 + np.e)

    funcs['F9'] = {
        'func': f9, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F10: Weierstrass ──────────────────────────────────────
    def f10(x):
        z  = rotations[10] @ (x - shifts[10])
        a, b, kmax = 0.5, 3.0, 20
        k  = np.arange(kmax+1)
        ak = a**k
        bk = b**k
        s  = np.sum([np.sum(ak*np.cos(2*np.pi*bk*(zi+0.5)))
                     for zi in z])
        c  = (kmax+1)*np.sum(ak*np.cos(2*np.pi*bk*0.5))
        return s - dim*c

    funcs['F10'] = {
        'func': f10, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F11: Griewank ─────────────────────────────────────────
    def f11(x):
        z = rotations[11] @ (x - shifts[11])
        return (np.sum(z**2)/4000
                - np.prod(np.cos(z/np.sqrt(
                    np.arange(1, dim+1)))) + 1)

    funcs['F11'] = {
        'func': f11, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F12: Schwefel ─────────────────────────────────────────
    def f12(x):
        z = x - shifts[12]
        return 0.5 + (np.sin(np.sqrt(np.sum(z**2)))**2-0.5) \
               / (1+0.001*np.sum(z**2))**2

    funcs['F12'] = {
        'func': f12, 'lb': -100, 'ub': 100,
        'dim': dim, 'type': 'Multimodal'
    }

    # ── F13-F20: Hybrid & Composition (simplified) ────────────
    # Use rotation+shift wrappers of base functions
    base_funcs = {
        13: lambda z: np.sum(z**2-10*np.cos(2*np.pi*z))+10*dim,
        14: lambda z: np.sum(z**2)/4000-np.prod(
            np.cos(z/np.sqrt(np.arange(1,dim+1))))+1,
        15: lambda z: np.sum(100*(z[1:]-z[:-1]**2)**2+(z[:-1]-1)**2),
        16: lambda z: np.sum(z**2-10*np.cos(2*np.pi*z))+10*dim,
        17: lambda z: (-20*np.exp(-0.2*np.sqrt(np.sum(z**2)/dim))
                       -np.exp(np.sum(np.cos(2*np.pi*z))/dim)+20+np.e),
        18: lambda z: np.sum(z**2)/4000-np.prod(
            np.cos(z/np.sqrt(np.arange(1,dim+1))))+1,
        19: lambda z: np.sum(z**2-10*np.cos(2*np.pi*z))+10*dim,
        20: lambda z: np.sum(100*(z[1:]-z[:-1]**2)**2+(z[:-1]-1)**2),
    }

    types = {13:'Hybrid',14:'Hybrid',15:'Hybrid',16:'Hybrid',
             17:'Hybrid',18:'Hybrid',19:'Composition',
             20:'Composition'}

    for i in range(13, 21):
        idx = i
        def make_f(ii):
            return lambda x: base_funcs[ii](
                rotations[ii] @ (x - shifts[ii]))
        funcs[f'F{i}'] = {
            'func': make_f(idx),
            'lb': -100, 'ub': 100,
            'dim': dim, 'type': types[idx]
        }

    # ── F21-F29: Composition functions ────────────────────────
    comp_bases = [
        lambda z: np.sum(z**2-10*np.cos(2*np.pi*z))+10*dim,
        lambda z: (-20*np.exp(-0.2*np.sqrt(np.sum(z**2)/dim))
                   -np.exp(np.sum(np.cos(2*np.pi*z))/dim)+20+np.e),
        lambda z: np.sum(z**2)/4000-np.prod(
            np.cos(z/np.sqrt(np.arange(1,dim+1))))+1,
    ]

    for i in range(21, 30):
        idx = i
        def make_comp(ii):
            n_comp = (ii-18) % 3 + 2
            def comp_f(x):
                z    = x - shifts[ii]
                vals = []
                for j in range(n_comp):
                    zr = rotations[ii] @ z
                    vals.append(comp_bases[j%3](zr))
                w = np.exp(-np.sum(z**2) / (2*dim*100**2))
                w = w / (np.sum(w) + 1e-10) \
                    if np.sum(w) > 0 else np.ones(n_comp)/n_comp
                return float(np.dot(
                    np.ones(n_comp)/n_comp, vals))
            return comp_f
        funcs[f'F{i}'] = {
            'func': make_comp(idx),
            'lb': -100, 'ub': 100,
            'dim': dim, 'type': 'Composition'
        }

    return funcs


def _make_rotation(dim):
    """Generate random orthogonal rotation matrix."""
    H = np.random.randn(dim, dim)
    Q, _ = np.linalg.qr(H)
    return Q


# ── Load for 30D and 50D ──────────────────────────────────────────
print('Building CEC 2017 function suites...')
CEC2017_30D = make_cec2017_functions(dim=30)
CEC2017_50D = make_cec2017_functions(dim=50)
print(f'✅ CEC 2017 30D: {len(CEC2017_30D)} functions')
print(f'✅ CEC 2017 50D: {len(CEC2017_50D)} functions')

Building CEC 2017 function suites...
✅ CEC 2017 30D: 28 functions
✅ CEC 2017 50D: 28 functions


In [48]:
### CELL 8b ####

def make_cec2017_fast(dim=30):
    """
    Optimised CEC 2017 — pre-computes all rotation matrices
    and shift vectors. Uses vectorised numpy operations
    throughout for maximum speed.
    """
    np.random.seed(2017)

    # Pre-compute ALL shifts and rotations once
    shifts    = {i: np.random.uniform(-80, 80, dim)
                 for i in range(1, 30)}
    rotations = {i: _make_rotation(dim)
                 for i in range(1, 30)}

    # Pre-compute rotated shifts for composition functions
    rot_shifts = {i: rotations[i] @ shifts[i]
                  for i in range(1, 30)}

    funcs = {}

    # ── Helper: apply rotation + shift ───────────────────────
    def transform(x, i):
        """Vectorised transform — single matrix multiply."""
        return rotations[i] @ (x - shifts[i])

    # ── F1: Bent Cigar ───────────────────────────────────────
    def f1(x):
        z = transform(x, 1)
        return z[0]**2 + 1e6 * np.dot(z[1:], z[1:])
    funcs['F1'] = {'func':f1,'lb':-100,'ub':100,
                   'dim':dim,'type':'Unimodal'}

    # ── F3: Zakharov ─────────────────────────────────────────
    _i3 = np.arange(1, dim+1) * 0.5
    def f3(x):
        z = transform(x, 3)
        s = np.dot(z, z)
        t = np.dot(_i3, z)
        return s + t**2 + t**4
    funcs['F3'] = {'func':f3,'lb':-100,'ub':100,
                   'dim':dim,'type':'Unimodal'}

    # ── F4: Rosenbrock ───────────────────────────────────────
    def f4(x):
        z = transform(x, 4) + 1
        return np.sum(100*(z[:-1]**2-z[1:])**2
                      + (z[:-1]-1)**2)
    funcs['F4'] = {'func':f4,'lb':-100,'ub':100,
                   'dim':dim,'type':'Multimodal'}

    # ── F5: Rastrigin ────────────────────────────────────────
    def f5(x):
        z = transform(x, 5)
        return 10*dim + np.dot(z,z) - 10*np.sum(
            np.cos(2*np.pi*z))
    funcs['F5'] = {'func':f5,'lb':-100,'ub':100,
                   'dim':dim,'type':'Multimodal'}

    # ── F6: Expanded Scaffer ─────────────────────────────────
    def f6(x):
        z  = transform(x, 6)
        zr = np.roll(z, -1)
        s  = z**2 + zr**2
        return np.sum(0.5 + (np.sin(np.sqrt(s))**2 - 0.5)
                      / (1 + 0.001*s)**2)
    funcs['F6'] = {'func':f6,'lb':-100,'ub':100,
                   'dim':dim,'type':'Multimodal'}

    # ── F7: Levy ─────────────────────────────────────────────
    def f7(x):
        z = transform(x, 7)
        w = 1 + (z-1)/4
        return (np.sin(np.pi*w[0])**2
                + np.dot((w[:-1]-1)**2,
                          1+10*np.sin(
                              np.pi*w[:-1]+1)**2)
                + (w[-1]-1)**2
                * (1+np.sin(2*np.pi*w[-1])**2))
    funcs['F7'] = {'func':f7,'lb':-100,'ub':100,
                   'dim':dim,'type':'Multimodal'}

    # ── F8: Modified Schwefel ────────────────────────────────
    _sw_offset = 4.209687462275036e+002
    _sw_base   = 4.189828872724338e+002 * dim
    def f8(x):
        z   = transform(x, 8) + _sw_offset
        s   = np.where(
            np.abs(z) <= 500,
            -z * np.sin(np.abs(z)**0.5),
            np.where(z > 500,
                (z-500)*np.sin(500**0.5)
                + (z-500)**2/(10000*dim),
                (z+500)*np.sin(500**0.5)
                + (z+500)**2/(10000*dim)))
        return np.sum(s) + _sw_base
    funcs['F8'] = {'func':f8,'lb':-100,'ub':100,
                   'dim':dim,'type':'Multimodal'}

    # ── F9: Ackley ───────────────────────────────────────────
    _inv_dim = 1.0 / dim
    def f9(x):
        z = transform(x, 9)
        return (-20*np.exp(-0.2*np.sqrt(
                    np.dot(z,z)*_inv_dim))
                - np.exp(np.sum(
                    np.cos(2*np.pi*z))*_inv_dim)
                + 20 + np.e)
    funcs['F9'] = {'func':f9,'lb':-100,'ub':100,
                   'dim':dim,'type':'Multimodal'}

    # ── F10: Weierstrass ─────────────────────────────────────
    _k10  = np.arange(21)
    _ak10 = 0.5**_k10
    _bk10 = 3.0**_k10
    _c10  = np.sum(_ak10*np.cos(2*np.pi*_bk10*0.5))
    def f10(x):
        z = transform(x, 10)
        # Vectorised over all z and k simultaneously
        zk = z[:,None] + 0.5  # (dim, 1)
        s  = np.sum(_ak10 * np.cos(
            2*np.pi*_bk10*zk))
        return s - dim*_c10
    funcs['F10'] = {'func':f10,'lb':-100,'ub':100,
                    'dim':dim,'type':'Multimodal'}

    # ── F11: Griewank ────────────────────────────────────────
    _sqrt_i11 = 1.0 / np.sqrt(np.arange(1, dim+1))
    def f11(x):
        z = transform(x, 11)
        return (np.dot(z,z)/4000
                - np.prod(np.cos(z*_sqrt_i11)) + 1)
    funcs['F11'] = {'func':f11,'lb':-100,'ub':100,
                    'dim':dim,'type':'Multimodal'}

    # ── F12: Schwefel 2.26 ───────────────────────────────────
    def f12(x):
        z = x - shifts[12]
        return (0.5 + (np.sin(np.sqrt(
                    np.dot(z,z)))**2 - 0.5)
                / (1 + 0.001*np.dot(z,z))**2)
    funcs['F12'] = {'func':f12,'lb':-100,'ub':100,
                    'dim':dim,'type':'Multimodal'}

    # ── F13-F20: Hybrid functions ────────────────────────────
    _base_funcs = [
        lambda z: (10*dim + np.dot(z,z)
                   - 10*np.sum(np.cos(2*np.pi*z))),
        lambda z: (np.dot(z,z)/4000
                   - np.prod(np.cos(
                       z*_sqrt_i11)) + 1),
        lambda z: np.sum(100*(z[:-1]**2-z[1:])**2
                         + (z[:-1]-1)**2),
        lambda z: (-20*np.exp(-0.2*np.sqrt(
                       np.dot(z,z)*_inv_dim))
                   - np.exp(np.sum(
                       np.cos(2*np.pi*z))*_inv_dim)
                   + 20 + np.e),
    ]

    types_h = {13:'Hybrid',14:'Hybrid',15:'Hybrid',
               16:'Hybrid',17:'Hybrid',18:'Hybrid',
               19:'Composition',20:'Composition'}

    for i in range(13, 21):
        def make_hybrid(ii):
            bf = _base_funcs[ii % len(_base_funcs)]
            return lambda x: bf(transform(x, ii))
        funcs[f'F{i}'] = {
            'func': make_hybrid(i),
            'lb':-100,'ub':100,
            'dim':dim,
            'type':types_h.get(i,'Hybrid')
        }

    # ── F21-F29: Composition functions ──────────────────────
    for i in range(21, 30):
        def make_comp(ii):
            n_comp = (ii-18) % 3 + 2
            bfs    = [_base_funcs[j % len(_base_funcs)]
                      for j in range(n_comp)]
            def comp_f(x):
                z    = x - shifts[ii]
                w    = np.exp(-np.dot(z,z)
                              / (2*dim*100**2))
                vals = np.array([
                    bfs[j](rotations[ii] @ z)
                    for j in range(n_comp)])
                return float(np.mean(vals))
            return comp_f
        funcs[f'F{i}'] = {
            'func': make_comp(i),
            'lb':-100,'ub':100,
            'dim':dim,
            'type':'Composition'
        }

    return funcs


# Rebuild with optimised versions
print('Building optimised CEC 2017 suites...')
CEC2017_30D = make_cec2017_fast(dim=30)
CEC2017_50D = make_cec2017_fast(dim=50)
print(f'✅ Optimised CEC 2017 30D: {len(CEC2017_30D)} functions')
print(f'✅ Optimised CEC 2017 50D: {len(CEC2017_50D)} functions')

# Quick speed test
print('\nSpeed test on F5 (Rastrigin):')
import time
start = time.time()
for _ in range(100):
    CEC2017_30D['F5']['func'](np.random.rand(30))
t = time.time() - start
print(f'  100 evaluations: {t:.3f}s')
print(f'  Per evaluation:  {t/100*1000:.2f}ms')

Building optimised CEC 2017 suites...
✅ Optimised CEC 2017 30D: 28 functions
✅ Optimised CEC 2017 50D: 28 functions

Speed test on F5 (Rastrigin):
  100 evaluations: 0.002s
  Per evaluation:  0.02ms


In [ ]:
#### CELL 9  #####

# ── CELL 9 — CEC 2017 30D RESUME ─────────────────────────────────
SEEDS         = list(range(42, 72))
ALGO_LIST     = list(ALGORITHM_REGISTRY.keys())
SAVE_PATH_30D = f'{BASE}/results/raw/cec2017_30d'

# Check what's already done
completed = get_completed_algos(SAVE_PATH_30D)
remaining = [a for a in ALGO_LIST if a not in completed]

print('CEC 2017 30D — RESUME MODE')
print(f'\nAlready completed ({len(completed)}):')
for a in completed:
    print(f'  ✅ {a}')
print(f'\nStill to run ({len(remaining)}):')
for a in remaining:
    print(f'  ⏳ {a}')
print(f'\nEstimated remaining time: '
      f'~{len(remaining)*13:.0f}-'
      f'{len(remaining)*20:.0f} minutes')
print()

# Run remaining only
cec30_results = run_benchmark_resume(
    funcs_dict = CEC2017_30D,
    algo_list  = ALGO_LIST,
    seeds      = SEEDS,
    save_path  = SAVE_PATH_30D,
    suite_name = 'CEC 2017 30D — Resume'
)

# Compile all results
cec30_summary, cec30_conv = compile_results(
    cec30_results, CEC2017_30D, ALGO_LIST)

with open(f'{SAVE_PATH_30D}/summary.json', 'w') as f:
    json.dump(cec30_summary, f, indent=2)

print('\n✅ CEC 2017 30D complete and saved')

CEC 2017 30D — RESUME MODE

Already completed (4):
  ✅ QSO
  ✅ PSO
  ✅ GA
  ✅ DE

Still to run (11):
  ⏳ GWO
  ⏳ WOA
  ⏳ SCA
  ⏳ HHO
  ⏳ MPA
  ⏳ BFO
  ⏳ QBSO
  ⏳ QBHO
  ⏳ DBO
  ⏳ POA
  ⏳ EVO

Estimated remaining time: ~143-220 minutes

Previously completed (4):
  ✅ QSO
  ✅ PSO
  ✅ GA
  ✅ DE

Remaining (11):
  ⏳ GWO
  ⏳ WOA
  ⏳ SCA
  ⏳ HHO
  ⏳ MPA
  ⏳ BFO
  ⏳ QBSO
  ⏳ QBHO
  ⏳ DBO
  ⏳ POA
  ⏳ EVO


CEC 2017 30D — Resume — Starting
  Algorithms : 11
  Functions  : 28
  Seeds      : 30
  Total runs : 9240

Running GWO... ✅ (1065.5s) | Elapsed: 17.8m | ETA: 177.6m
Running WOA... ✅ (818.3s) | Elapsed: 31.4m | ETA: 141.3m
Running SCA... ✅ (693.5s) | Elapsed: 43.0m | ETA: 114.5m
Running HHO... ✅ (1307.0s) | Elapsed: 64.7m | ETA: 113.3m
Running MPA... ✅ (747.5s) | Elapsed: 77.2m | ETA: 92.6m
Running BFO... ✅ (976.9s) | Elapsed: 93.5m | ETA: 77.9m
Running QBSO... ✅ (701.0s) | Elapsed: 105.2m | ETA: 60.1m
Running QBHO... ✅ (647.7s) | Elapsed: 116.0m | ETA: 43.5m
Running DBO... ✅ (642.3s) | Elaps

In [23]:
### CELL 10  ####

# ── CELL 10 — CEC 2017 50D ───────────────────────────────────────
SEEDS         = list(range(42, 72))
ALGO_LIST     = list(ALGORITHM_REGISTRY.keys())
SAVE_PATH_50D = f'{BASE}/results/raw/cec2017_50d'

# Check what's already done
completed = get_completed_algos(SAVE_PATH_50D)
remaining = [a for a in ALGO_LIST if a not in completed]

print('CEC 2017 50D — Starting')
print(f'\nAlready completed ({len(completed)}):')
for a in completed:
    print(f'  ✅ {a}')
print(f'\nStill to run ({len(remaining)}):')
for a in remaining:
    print(f'  ⏳ {a}')
print()

cec50_results = run_benchmark_resume(
    funcs_dict = CEC2017_50D,
    algo_list  = ALGO_LIST,
    seeds      = SEEDS,
    save_path  = SAVE_PATH_50D,
    suite_name = 'CEC 2017 50D'
)

cec50_summary, cec50_conv = compile_results(
    cec50_results, CEC2017_50D, ALGO_LIST)

with open(f'{SAVE_PATH_50D}/summary.json', 'w') as f:
    json.dump(cec50_summary, f, indent=2)

print('\n✅ CEC 2017 50D complete and saved')

CEC 2017 50D — Starting

Already completed (7):
  ✅ QSO
  ✅ PSO
  ✅ GA
  ✅ DE
  ✅ GWO
  ✅ WOA
  ✅ SCA

Still to run (8):
  ⏳ HHO
  ⏳ MPA
  ⏳ BFO
  ⏳ QBSO
  ⏳ QBHO
  ⏳ DBO
  ⏳ POA
  ⏳ EVO

Previously completed (7):
  ✅ QSO
  ✅ PSO
  ✅ GA
  ✅ DE
  ✅ GWO
  ✅ WOA
  ✅ SCA

Remaining (8):
  ⏳ HHO
  ⏳ MPA
  ⏳ BFO
  ⏳ QBSO
  ⏳ QBHO
  ⏳ DBO
  ⏳ POA
  ⏳ EVO


CEC 2017 50D — Starting
  Algorithms : 8
  Functions  : 28
  Seeds      : 30
  Total runs : 6720

Running HHO... ✅ (1543.1s) | Elapsed: 25.7m | ETA: 180.0m
Running MPA... ✅ (923.3s) | Elapsed: 41.1m | ETA: 123.3m
Running BFO... ✅ (1340.7s) | Elapsed: 63.5m | ETA: 105.8m
Running QBSO... ✅ (816.5s) | Elapsed: 77.1m | ETA: 77.1m
Running QBHO... ✅ (776.1s) | Elapsed: 90.0m | ETA: 54.0m
Running DBO... ✅ (771.2s) | Elapsed: 102.8m | ETA: 34.3m
Running POA... ✅ (1691.8s) | Elapsed: 131.0m | ETA: 18.7m
Running EVO... ✅ (837.2s) | Elapsed: 145.0m | ETA: 0.0m

✅ CEC 2017 50D complete
   Total time: 145.1 minutes
   Results saved: /content/drive/MyDri

In [50]:
#### CELL 11 ######

# ── Preview CEC 2017 Results — 30D and 50D ────────────────────────
import json
import numpy as np

BASE          = '/content/drive/MyDrive/QSO_Research'
SAVE_PATH_30D = f'{BASE}/results/raw/cec2017_30d'
SAVE_PATH_50D = f'{BASE}/results/raw/cec2017_50d'
ALGO_LIST     = list(ALGORITHM_REGISTRY.keys())


def compute_friedman(summary, save_path):
    """Load results and compute Friedman ranking."""
    # Load raw results to get function list
    with open(f'{save_path}/all_results.json', 'r') as f:
        raw = json.load(f)

    func_ids = sorted(list(set(r['func'] for r in raw)))

    rank_matrix = np.zeros(
        (len(ALGO_LIST), len(func_ids)))

    for j, func_id in enumerate(func_ids):
        means = []
        for algo in ALGO_LIST:
            if (func_id in summary.get(algo, {}) and
                    summary[algo][func_id]['mean']
                    != float('inf')):
                means.append(
                    summary[algo][func_id]['mean'])
            else:
                means.append(float('inf'))

        order = np.argsort(means)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(ALGO_LIST)+1)
        rank_matrix[:, j] = ranks

    mean_ranks = np.mean(rank_matrix, axis=1)
    return mean_ranks, rank_matrix, func_ids


def print_ranking(mean_ranks, rank_matrix,
                  suite_name, algo_list):
    """Print formatted Friedman ranking table."""
    sorted_idx = np.argsort(mean_ranks)
    qso_idx    = algo_list.index('QSO')
    qso_ranks  = rank_matrix[qso_idx]

    print(f'\n{"="*60}')
    print(f'{suite_name} — FRIEDMAN RANKING')
    print(f'{"="*60}')
    print(f'\n{"Rank":<6} {"Algorithm":<8} '
          f'{"Mean Friedman Rank":<22} {"W/D/L vs QSO"}')
    print('-'*58)

    for display_rank, idx in enumerate(sorted_idx, 1):
        algo  = algo_list[idx]
        mrank = mean_ranks[idx]

        if algo == 'QSO':
            wdl = '—'
        else:
            algo_ranks = rank_matrix[idx]
            w = int(np.sum(algo_ranks > qso_ranks))
            d = int(np.sum(algo_ranks == qso_ranks))
            l = int(np.sum(algo_ranks < qso_ranks))
            wdl = f'{w}W/{d}D/{l}L'

        marker = ' ← QSO' if algo == 'QSO' else ''
        print(f'{display_rank:<6} {algo:<8} '
              f'{mrank:<22.2f} {wdl}{marker}')

    # QSO detailed stats
    qso_rank_val = mean_ranks[qso_idx]
    print(f'\n  QSO Friedman rank:      {qso_rank_val:.2f}')
    print(f'  Functions ranked 1st:   '
          f'{int(np.sum(qso_ranks == 1))}')
    print(f'  Functions ranked top 3: '
          f'{int(np.sum(qso_ranks <= 3))}')
    print(f'  Functions ranked top 5: '
          f'{int(np.sum(qso_ranks <= 5))}')


# ── Load summaries ────────────────────────────────────────────────
print('Loading results from Drive...')

with open(f'{SAVE_PATH_30D}/summary.json', 'r') as f:
    cec30_summary = json.load(f)

with open(f'{SAVE_PATH_50D}/summary.json', 'r') as f:
    cec50_summary = json.load(f)

print('✅ Both summaries loaded')

# ── Compute rankings ──────────────────────────────────────────────
ranks_30d, matrix_30d, funcs_30d = compute_friedman(
    cec30_summary, SAVE_PATH_30D)

ranks_50d, matrix_50d, funcs_50d = compute_friedman(
    cec50_summary, SAVE_PATH_50D)

# ── Print rankings ────────────────────────────────────────────────
print_ranking(ranks_30d, matrix_30d,
              'CEC 2017 30D', ALGO_LIST)

print_ranking(ranks_50d, matrix_50d,
              'CEC 2017 50D', ALGO_LIST)

# ── Cross-dimension comparison ────────────────────────────────────
qso_idx = ALGO_LIST.index('QSO')

print(f'\n{"="*60}')
print('QSO SCALABILITY — 30D vs 50D')
print(f'{"="*60}')
print(f'\n{"Algorithm":<8} {"30D Rank":<12} '
      f'{"50D Rank":<12} {"Trend"}')
print('-'*45)

sorted_by_30d = np.argsort(ranks_30d)
for idx in sorted_by_30d:
    algo  = ALGO_LIST[idx]
    r30   = ranks_30d[idx]
    r50   = ranks_50d[idx]
    diff  = r30 - r50
    if diff > 0.3:
        trend = f'⬆️  +{diff:.2f} (improves)'
    elif diff < -0.3:
        trend = f'⬇️  {diff:.2f} (worsens)'
    else:
        trend = '➡️  stable'

    marker = ' ← QSO' if algo == 'QSO' else ''
    print(f'{algo:<8} {r30:<12.2f} '
          f'{r50:<12.2f} {trend}{marker}')

Loading results from Drive...
✅ Both summaries loaded

CEC 2017 30D — FRIEDMAN RANKING

Rank   Algorithm Mean Friedman Rank     W/D/L vs QSO
----------------------------------------------------------
1      DE       4.75                   11W/0D/17L
2      BFO      4.82                   19W/0D/9L
3      QSO      4.89                   — ← QSO
4      HHO      5.11                   19W/0D/9L
5      QBSO     5.68                   19W/0D/9L
6      WOA      6.39                   19W/0D/9L
7      GA       6.64                   25W/0D/3L
8      PSO      7.29                   23W/0D/5L
9      DBO      7.86                   22W/0D/6L
10     GWO      8.25                   19W/0D/9L
11     POA      9.11                   19W/0D/9L
12     EVO      11.14                  22W/0D/6L
13     QBHO     11.68                  22W/0D/6L
14     SCA      12.57                  21W/0D/7L
15     MPA      13.82                  23W/0D/5L
16     GJO      16.00                  28W/0D/0L

  QSO Friedman r

In [ ]:
import os

SAVE_PATH = f'{BASE}/results/raw/classical'

print('Checking checkpoints on Drive...')
for algo in ALGORITHM_REGISTRY:
    chk = f'{SAVE_PATH}/checkpoint_{algo}.json'
    exists = os.path.exists(chk)
    status = '✅ saved' if exists else '❌ missing'
    print(f'  {algo:<6}: {status}')

Checking checkpoints on Drive...
  QSO   : ✅ saved
  PSO   : ✅ saved
  GA    : ✅ saved
  DE    : ✅ saved
  GWO   : ✅ saved
  WOA   : ✅ saved
  SCA   : ✅ saved
  HHO   : ✅ saved
  MPA   : ✅ saved
  BFO   : ✅ saved
  QBSO  : ✅ saved
  QBHO  : ✅ saved
  DBO   : ✅ saved
  POA   : ✅ saved
  EVO   : ✅ saved


In [ ]:
# Quick speed test — compare old vs new
import time

sphere = lambda x: np.sum(x**2)

print('Speed test — QSO on Sphere 30D, 5 seeds:')
start = time.time()
for seed in range(42, 47):
    qso_func(sphere, -100, 100, dim=30, seed=seed)
elapsed = time.time() - start
print(f'  5 runs: {elapsed:.1f}s')
print(f'  Per run: {elapsed/5:.1f}s')
print(f'  Projected Classical 23 time: '
      f'{elapsed/5*23*30/60:.1f} minutes')

Speed test — QSO on Sphere 30D, 5 seeds:
  5 runs: 2.9s
  Per run: 0.6s
  Projected Classical 23 time: 6.8 minutes


In [ ]:
# Verify fast QSO is loaded
import time

print('Verifying fast QSO is active...')
start = time.time()
f, _, _ = run_algorithm('QSO', sphere, -100, 100,
                          dim=30, seed=42)
elapsed = time.time() - start

print(f'  Single run time: {elapsed:.2f}s')

if elapsed < 2.0:
    print('  ✅ Fast QSO confirmed — good to go')
else:
    print('  ⚠️  Still slow — reload QSO from Drive')
    print('  Rerun Cell 2 to reload')

# Also verify all competitors load
print('\nVerifying all 15 algorithms...')
errors = []
for algo in ALGORITHM_REGISTRY:
    try:
        start = time.time()
        f, _, _ = run_algorithm(
            algo, sphere, -100, 100, dim=30, seed=42)
        t = time.time() - start
        print(f'  ✅ {algo:<6}: {f:.4e} ({t:.1f}s)')
    except Exception as e:
        print(f'  ❌ {algo:<6}: {e}')
        errors.append(algo)

if errors:
    print(f'\n⚠️  Fix before proceeding: {errors}')
else:
    print(f'\n✅ All algorithms verified — ready to run')

Verifying fast QSO is active...
  Single run time: 0.27s
  ✅ Fast QSO confirmed — good to go

Verifying all 15 algorithms...
  ✅ QSO   : 8.4955e-02 (0.3s)
  ✅ PSO   : 1.1721e-05 (0.2s)
  ✅ GA    : 6.9737e+00 (1.2s)
  ✅ DE    : 3.1691e-01 (0.8s)
  ✅ GWO   : 1.3550e-31 (0.6s)
  ✅ WOA   : 3.7246e-07 (0.4s)
  ✅ SCA   : 5.3338e-12 (0.3s)
  ✅ HHO   : 1.4046e-84 (0.6s)
  ✅ MPA   : 1.4499e-02 (0.4s)
  ✅ BFO   : 4.8163e-01 (0.3s)
  ✅ QBSO  : 2.8946e+00 (0.3s)
  ✅ QBHO  : 8.8982e+03 (0.3s)
  ✅ DBO   : 4.4626e-04 (0.3s)
  ✅ POA   : 1.9721e-123 (0.6s)
  ✅ EVO   : 1.7627e+03 (0.3s)

✅ All algorithms verified — ready to run


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json

BASE      = '/content/drive/MyDrive/QSO_Research'
SAVE_PATH = f'{BASE}/results/raw/cec2017_30d'

print('CEC 2017 30D — Checkpoint Status')
print('='*45)

algos = ['QSO','PSO','GA','DE','GWO','WOA','SCA',
         'HHO','MPA','BFO','QBSO','QBHO','DBO','POA','EVO']

completed = []
missing   = []

for algo in algos:
    chk  = f'{SAVE_PATH}/checkpoint_{algo}.json'
    if os.path.exists(chk):
        # Verify file is valid and not empty
        try:
            with open(chk) as f:
                data = json.load(f)
            n_runs = len(data)
            completed.append(algo)
            print(f'  ✅ {algo:<6}: {n_runs} runs saved')
        except:
            missing.append(algo)
            print(f'  ⚠️  {algo:<6}: corrupted checkpoint')
    else:
        missing.append(algo)
        print(f'  ❌ {algo:<6}: not saved')

print(f'\nCompleted: {len(completed)}/15')
print(f'Missing:   {len(missing)}/15')
print(f'\nMissing algorithms: {missing}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CEC 2017 30D — Checkpoint Status
  ✅ QSO   : 840 runs saved
  ✅ PSO   : 840 runs saved
  ✅ GA    : 840 runs saved
  ✅ DE    : 840 runs saved
  ✅ GWO   : 840 runs saved
  ✅ WOA   : 840 runs saved
  ✅ SCA   : 840 runs saved
  ✅ HHO   : 840 runs saved
  ✅ MPA   : 840 runs saved
  ✅ BFO   : 840 runs saved
  ✅ QBSO  : 840 runs saved
  ✅ QBHO  : 840 runs saved
  ✅ DBO   : 840 runs saved
  ✅ POA   : 840 runs saved
  ✅ EVO   : 840 runs saved

Completed: 15/15
Missing:   0/15

Missing algorithms: []


In [ ]:
import os
BASE      = '/content/drive/MyDrive/QSO_Research'
SAVE_PATH = f'{BASE}/results/raw/cec2017_30d'

for algo in ['QSO', 'PSO', 'GA']:
    chk = f'{SAVE_PATH}/checkpoint_{algo}.json'
    print(f'{algo}: {"✅ found" if os.path.exists(chk) else "❌ missing"}')

QSO: ✅ found
PSO: ✅ found
GA: ✅ found


In [ ]:
# Run GJO on all suites
SEEDS    = list(range(42, 72))
GJO_ONLY = ['GJO']

# Classical 23
gjo_classical = run_benchmark_suite(
    CLASSICAL_FUNCS, GJO_ONLY, SEEDS,
    f'{BASE}/results/raw/classical',
    'GJO Classical 23'
)

# CEC 2017 30D
gjo_cec30 = run_benchmark_suite(
    CEC2017_30D, GJO_ONLY, SEEDS,
    f'{BASE}/results/raw/cec2017_30d',
    'GJO CEC 2017 30D'
)

# CEC 2017 50D
gjo_cec50 = run_benchmark_suite(
    CEC2017_50D, GJO_ONLY, SEEDS,
    f'{BASE}/results/raw/cec2017_50d',
    'GJO CEC 2017 50D'
)

In [49]:
# ── Run GJO on all benchmark suites ──────────────────────────────
import time

SEEDS     = list(range(42, 72))
GJO_ONLY  = ['GJO']

print('='*55)
print('RUNNING GJO ON ALL BENCHMARK SUITES')
print('='*55)
print(f'  Seeds: {len(SEEDS)}')
print(f'  Estimated time: ~15-25 minutes total\n')

# ── Classical 23 ──────────────────────────────────────────────────
print('1. Classical 23...')
start = time.time()
run_benchmark_suite(
    funcs_dict = CLASSICAL_FUNCS,
    algo_list  = GJO_ONLY,
    seeds      = SEEDS,
    save_path  = f'{BASE}/results/raw/classical',
    suite_name = 'GJO Classical 23'
)
print(f'   Done in {(time.time()-start)/60:.1f} mins\n')

# ── CEC 2017 30D ──────────────────────────────────────────────────
print('2. CEC 2017 30D...')
start = time.time()
run_benchmark_suite(
    funcs_dict = CEC2017_30D,
    algo_list  = GJO_ONLY,
    seeds      = SEEDS,
    save_path  = f'{BASE}/results/raw/cec2017_30d',
    suite_name = 'GJO CEC 2017 30D'
)
print(f'   Done in {(time.time()-start)/60:.1f} mins\n')

# ── CEC 2017 50D ──────────────────────────────────────────────────
print('3. CEC 2017 50D...')
start = time.time()
run_benchmark_suite(
    funcs_dict = CEC2017_50D,
    algo_list  = GJO_ONLY,
    seeds      = SEEDS,
    save_path  = f'{BASE}/results/raw/cec2017_50d',
    suite_name = 'GJO CEC 2017 50D'
)
print(f'   Done in {(time.time()-start)/60:.1f} mins\n')

print('='*55)
print('✅ GJO complete on all suites')
print('   Rerun Cell 11 to see updated rankings')
print('='*55)

RUNNING GJO ON ALL BENCHMARK SUITES
  Seeds: 30
  Estimated time: ~15-25 minutes total

1. Classical 23...

GJO Classical 23 — Starting
  Algorithms : 1
  Functions  : 23
  Seeds      : 30
  Total runs : 690

Running GJO... ✅ (622.9s) | Elapsed: 10.4m | ETA: 0.0m

✅ GJO Classical 23 complete
   Total time: 10.4 minutes
   Results saved: /content/drive/MyDrive/QSO_Research/results/raw/classical/all_results.json
   Done in 10.4 mins

2. CEC 2017 30D...

GJO CEC 2017 30D — Starting
  Algorithms : 1
  Functions  : 28
  Seeds      : 30
  Total runs : 840

Running GJO... ✅ (839.1s) | Elapsed: 14.0m | ETA: 0.0m

✅ GJO CEC 2017 30D complete
   Total time: 14.0 minutes
   Results saved: /content/drive/MyDrive/QSO_Research/results/raw/cec2017_30d/all_results.json
   Done in 14.0 mins

3. CEC 2017 50D...

GJO CEC 2017 50D — Starting
  Algorithms : 1
  Functions  : 28
  Seeds      : 30
  Total runs : 840

Running GJO... ✅ (906.7s) | Elapsed: 15.1m | ETA: 0.0m

✅ GJO CEC 2017 50D complete
   Total 

In [51]:
# Quick GJO diagnosis
import numpy as np

sphere     = lambda x: np.sum(x**2)
rastrigin  = lambda x: (10*len(x)
             + np.sum(x**2
             - 10*np.cos(2*np.pi*x)))

print('GJO Diagnosis — Single Seed Tests')
print('='*45)

for fname, func, lb, ub in [
    ('Sphere',    sphere,    -100,  100),
    ('Rastrigin', rastrigin, -5.12, 5.12),
]:
    results = []
    for seed in range(42, 52):
        f, _, c = gjo(func, lb, ub,
                      dim=30, seed=seed)
        results.append(f)

    print(f'\n{fname} 30D (10 seeds):')
    print(f'  Mean:  {np.mean(results):.4e}')
    print(f'  Best:  {np.min(results):.4e}')
    print(f'  Worst: {np.max(results):.4e}')
    print(f'  Std:   {np.std(results):.4e}')

# Compare with GWO as reference
print('\nGWO reference (should be ~1e-31 on Sphere):')
f, _, _ = gwo(sphere, -100, 100, dim=30, seed=42)
print(f'  GWO Sphere: {f:.4e}')

print('\nGJO reference:')
f, _, _ = gjo(sphere, -100, 100, dim=30, seed=42)
print(f'  GJO Sphere: {f:.4e}')

GJO Diagnosis — Single Seed Tests

Sphere 30D (10 seeds):
  Mean:  1.6554e-127
  Best:  2.5637e-136
  Worst: 1.2126e-126
  Std:   3.5671e-127

Rastrigin 30D (10 seeds):
  Mean:  0.0000e+00
  Best:  0.0000e+00
  Worst: 0.0000e+00
  Std:   0.0000e+00

GWO reference (should be ~1e-31 on Sphere):
  GWO Sphere: 1.3550e-31

GJO reference:
  GJO Sphere: 4.7636e-134
